In [1]:
from sqlalchemy.engine import URL

db_url = URL.create(
    drivername="postgresql+psycopg",
    username="postgres",
    password="password",
    host="localhost",
    port=5555,
    database="similarity_search_service_db",
)

In [2]:
from typing import List
from pgvector.sqlalchemy import Vector
from sqlalchemy import Integer, String
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column


# Create the base class for the table definition
class Base(DeclarativeBase):
    __abstract__ = True


# Create the table definition
class Images(Base):
    __tablename__ = "images"
    VECTOR_LENGTH = 512

    # primary key
    id: Mapped[int] = mapped_column(Integer, primary_key=True)
    # image path - we will use it to store the path to the image file, after similarity search we can use it to retrieve the image and display it
    image_path: Mapped[str] = mapped_column(String(256))
    # image embedding - we will store the image embedding in this column, the image embedding is a list of 512 floats this is the output of the sentence transformer model
    image_embedding: Mapped[List[float]] = mapped_column(Vector(VECTOR_LENGTH))

In [3]:
from sqlalchemy import create_engine

engine = create_engine(db_url)

In [4]:
Base.metadata.create_all(engine)

In [7]:
from sqlalchemy.orm import Session
from sqlalchemy import Engine, select
import numpy as np


# reusable function to insert data into the table
def insert_image(engine: Engine, image_path: str, image_embedding: list[float]):
    with Session(engine) as session:
        # create the image object
        image = Images(image_path=image_path, image_embedding=image_embedding)
        # add the image object to the session
        session.add(image)
        # commit the transaction
        session.commit()


# insert some data into the table
N = 100
for i in range(N):
    image_path = f"image_{i}.jpg"
    image_embedding = np.random.rand(512).tolist()
    insert_image(engine, image_path, image_embedding)

# select first image from the table
with Session(engine) as session:
    image = session.query(Images).first()


# Note: pgvector exposes *distances*, not similarities (lower = more similar).
# To rank, sort ascending by `cosine_distance`. To filter by a similarity threshold,
# use `1 - cosine_distance > threshold`.


# order the K images by ascending cosine distance to the first image (smallest distance = most similar)
def find_k_images(engine: Engine, k: int, original_image: Images) -> list[Images]:
    with Session(engine) as session:
        # execution_options={"prebuffer_rows": True} is used to prebuffer the rows, this is useful when we want to fetch the rows in chunks and return them after session is closed
        result = (
            session.execute(
                select(Images)
                .order_by(
                    Images.image_embedding.cosine_distance(
                        original_image.image_embedding
                    )
                )
                .limit(k),
                execution_options={"prebuffer_rows": True},
            )
            .scalars()
            .all()
        )
        return result


# find the 10 most similar images to the first image
k = 10
similar_images = find_k_images(engine, k, image)

In [8]:
# find images with similarity to the original above the threshold (similarity = 1 - cosine_distance)
def find_images_with_similarity_score_greater_than(
    engine: Engine, similarity_score: float, original_image: Images
) -> list[Images]:
    with Session(engine) as session:
        result = (
            session.execute(
                select(Images).filter(
                    (
                        1
                        - Images.image_embedding.cosine_distance(
                            original_image.image_embedding
                        )
                    )
                    > similarity_score
                ),
                execution_options={"prebuffer_rows": True},
            )
            .scalars()
            .all()
        )
        return result

In [13]:
from datasets import load_dataset

dataset = load_dataset("FronkonGames/steam-games-dataset")

# get columns names and types
columns = dataset["train"].features
print(columns)

columns_to_keep = [
    "name",
    "windows",
    "linux",
    "mac",
    "detailed_description",
    "supported_languages",
    "price",
]

N = 10000  # you can adjust this number
dataset = dataset["train"].select_columns(columns_to_keep).select(range(N))

{'appID': Value('string'), 'name': Value('string'), 'release_date': Value('string'), 'estimated_owners': Value('string'), 'peak_ccu': Value('int64'), 'required_age': Value('int64'), 'price': Value('float64'), 'dlc_count': Value('int64'), 'detailed_description': Value('string'), 'short_description': Value('string'), 'supported_languages': List(Value('string')), 'full_audio_languages': List(Value('string')), 'reviews': Value('string'), 'header_image': Value('string'), 'website': Value('string'), 'support_url': Value('string'), 'support_email': Value('string'), 'windows': Value('bool'), 'mac': Value('bool'), 'linux': Value('bool'), 'metacritic_score': Value('int64'), 'metacritic_url': Value('string'), 'user_score': Value('int64'), 'positive': Value('int64'), 'negative': Value('int64'), 'score_rank': Value('string'), 'achievements': Value('int64'), 'recommendations': Value('int64'), 'notes': Value('string'), 'average_playtime_forever': Value('int64'), 'average_playtime_2weeks': Value('int6

In [9]:
from sqlalchemy import Integer, Float, Boolean, Text


class Games(Base):
    __tablename__ = "games"
    __table_args__ = {"extend_existing": True}

    # the vector size produced by the model taken from documentation https://huggingface.co/sentence-transformers/distiluse-base-multilingual-cased-v2
    VECTOR_LENGTH = 512

    id: Mapped[int] = mapped_column(Integer, primary_key=True)
    name: Mapped[str] = mapped_column(String(256))
    # `Text` is an unbounded string in Postgres — game descriptions can run a few KB
    description: Mapped[str] = mapped_column(Text)
    windows: Mapped[bool] = mapped_column(Boolean)
    linux: Mapped[bool] = mapped_column(Boolean)
    mac: Mapped[bool] = mapped_column(Boolean)
    price: Mapped[float] = mapped_column(Float)
    game_description_embedding: Mapped[list[float]] = mapped_column(
        Vector(VECTOR_LENGTH)
    )


Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

In [10]:
from sentence_transformers import SentenceTransformer


checkpoint = "distiluse-base-multilingual-cased-v2"
model = SentenceTransformer(checkpoint)


def generate_embeddings(text: str) -> list[float]:
    return model.encode(text)

In [11]:
from tqdm import tqdm


def insert_games(engine, dataset):
    with tqdm(total=len(dataset)) as pbar:
        for i, game in enumerate(dataset):
            game_description = game["detailed_description"] or ""
            game_embedding = generate_embeddings(game_description)
            name, windows, linux, mac, price = (
                game["name"],
                game["windows"],
                game["linux"],
                game["mac"],
                game["price"],
            )
            # keep rows with the required fields; windows/linux/mac are platform flags (True/False),
            # not missing-data signals — don't gate the insert on them or you discard most of the dataset
            if name and game_description and price is not None:
                game = Games(
                    name=game["name"],
                    description=game_description,
                    windows=game["windows"],
                    linux=game["linux"],
                    mac=game["mac"],
                    price=game["price"],
                    game_description_embedding=game_embedding,
                )
                with Session(engine) as session:
                    session.add(game)
                    session.commit()
            pbar.update(1)

In [14]:
insert_games(engine, dataset)

100%|██████████| 10000/10000 [02:26<00:00, 68.16it/s]


In [16]:
from typing import Optional


def find_game(
    engine: Engine,
    game_description: str,
    windows: Optional[bool] = None,
    linux: Optional[bool] = None,
    mac: Optional[bool] = None,
    price: Optional[int] = None,
):
    with Session(engine) as session:
        game_embedding = generate_embeddings(game_description)

        query = select(Games).order_by(
            Games.game_description_embedding.cosine_distance(game_embedding)
        )

        if price:
            query = query.filter(Games.price <= price)
        if windows:
            query = query.filter(Games.windows == True)
        if linux:
            query = query.filter(Games.linux == True)
        if mac:
            query = query.filter(Games.mac == True)

        result = session.execute(query, execution_options={"prebuffer_rows": True})
        game = result.scalars().first()

        return game

In [17]:
game = find_game(engine, "This is a game about a hero who saves the world", price=10)
print(f"Game: {game.name}")
print(f"Description: {game.description}")

game = find_game(engine, game_description="Home decorating", price=20)
print(f"Game: {game.name}")
print(f"Description: {game.description}")

Game: FadeZone 消逝之地
Description: It's an adventure role-playing game, you've been summoned to this world as a Hero, along the way you will encounter various puzzles and NPCs, and eventually defeat Mayor(the mayor) assimilated with the devil, save this world. * But, only half of this description is truth. If someone wanna translate this game to other language. You are welcome! ——————————Spoilers Warning—————————— Players will control a silent, thoughtless, weakness 'Hero' .Come to the town. With the Black cat, complete the puzzles, and kill the Devil of Mayor. But, it is not the end. Players can get more stories and learn the truth of the world in the playing again and again. The so-called 'Hero fight against the Devil' is only a forced disguise, hidden behind the disguise is indeed conspiracy and rights? And the truth of everything will be very cruel, so-called rights are useless, so-called conspiracy are foolish disguises. 'The world is just a box. Inside box, there's another box. Box

In [18]:
game = find_game(engine, game_description="Home decorating", mac=True, price=5)
print(f"Game: {game.name}")
print(f"Description: {game.description}")

Game: A House of Many Doors
Description: Welcome to the House. You are not welcome. Explore the House, a parasite dimension that steals from other worlds, in a train that scuttles on mechanical legs. Uncover secrets. Open locked doors. Lead a crew of dysfunctional characters. Write procedurally-generated poems. Fight in turn-based combat. Explore a strange new setting, dripping with atmosphere, crusted with lore. Escape. Escape. Escape. You are an explorer, poet and spy, launching yourself into the unknown in search of adventure. Rig an election in the city of the dead. Visit a village lit by the burning corpse of a god (careful not to inhale the holy smoke). Sell your teeth to skittering spider-things for a moment in their library. Over 90 bizarre locations await discovery in the dust and the dark. A House of Many Doors is a 2D exploration RPG that takes inspiration from Sunless Sea, China Mieville, Planescape: Torment and Italo Calvino's Invisible Cities. It features over 300,000 wor

In [19]:
from pymilvus import MilvusClient

milvus_client = MilvusClient("./milvus_rag.db")

In [20]:
from pymilvus import FieldSchema, DataType, CollectionSchema

VECTOR_LENGTH = 768  # check the dimensionality for Silver Retriever Base (v1.1) model

id_field = FieldSchema(
    name="id", dtype=DataType.INT64, is_primary=True, description="Primary id"
)
text = FieldSchema(
    name="text", dtype=DataType.VARCHAR, max_length=4096, description="Page text"
)
embedding_text = FieldSchema(
    "embedding",
    dtype=DataType.FLOAT_VECTOR,
    dim=VECTOR_LENGTH,
    description="Embedded text",
)

fields = [id_field, text, embedding_text]

schema = CollectionSchema(
    fields=fields,
    auto_id=True,
    enable_dynamic_field=True,
    description="RAG Texts collection",
)

In [21]:
COLLECTION_NAME = "rag_texts_and_embeddings"

milvus_client.create_collection(collection_name=COLLECTION_NAME, schema=schema)

index_params = milvus_client.prepare_index_params()

index_params.add_index(
    field_name="embedding",
    index_type="HNSW",
    metric_type="L2",
    params={"M": 4, "efConstruction": 64},  # lower values for speed
)

milvus_client.create_index(collection_name=COLLECTION_NAME, index_params=index_params)

# checkout our collection
print(milvus_client.list_collections())

# describe our collection
print(milvus_client.describe_collection(COLLECTION_NAME))

['rag_texts_and_embeddings']
{'collection_name': 'rag_texts_and_embeddings', 'auto_id': True, 'num_shards': 1, 'description': '', 'fields': [{'field_id': 0, 'name': 'id', 'description': '', 'type': <DataType.INT64: 5>, 'params': {}, 'auto_id': True, 'is_primary': True}, {'field_id': 0, 'name': 'text', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 4096}}, {'field_id': 0, 'name': 'embedding', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 768}}], 'functions': [], 'aliases': [], 'collection_id': 0, 'consistency_level': 0, 'properties': {}, 'num_partitions': 1, 'enable_dynamic_field': True}


In [22]:
# define data source and destination
## the document origin destination from which document will be downloaded
pdf_url = "https://www.iab.org.pl/wp-content/uploads/2024/04/Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.pdf"

## local destination of the document
file_name = "Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.pdf"

## local destination of the processed document
file_json = "Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.json"

## local destination of the embedded pages of the document
embeddings_json = "Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska-Embeddings.json"

## local destination of all above local required files
data_dir = "./data"

In [23]:
# download data
import os
import requests


def download_pdf_data(pdf_url: str, file_name: str) -> None:
    response = requests.get(pdf_url, stream=True)
    os.makedirs(data_dir, exist_ok=True)
    with open(os.path.join(data_dir, file_name), "wb") as file:
        for block in response.iter_content(chunk_size=1024):
            if block:
                file.write(block)


download_pdf_data(pdf_url, file_name)

In [24]:
# prepare data

import fitz
import json


def extract_pdf_text(file_name, file_json):
    document = fitz.open(os.path.join(data_dir, file_name))
    pages = []

    for page_num in range(len(document)):
        page = document.load_page(page_num)
        page_text = page.get_text()
        pages.append({"page_num": page_num, "text": page_text})

    with open(os.path.join(data_dir, file_json), "w") as file:
        json.dump(pages, file, indent=4, ensure_ascii=False)


extract_pdf_text(file_name, file_json)

In [25]:
# vectorize data

import torch
import numpy as np
from sentence_transformers import SentenceTransformer


def generate_embeddings(file_json, embeddings_json, model):
    pages = []
    with open(os.path.join(data_dir, file_json), "r") as file:
        data = json.load(file)

    for page in data:
        pages.append(page["text"])

    embeddings = model.encode(pages)

    embeddings_paginated = []
    for page_num in range(len(embeddings)):
        embeddings_paginated.append(
            {"page_num": page_num, "embedding": embeddings[page_num].tolist()}
        )

    with open(os.path.join(data_dir, embeddings_json), "w") as file:
        json.dump(embeddings_paginated, file, indent=4, ensure_ascii=False)


model_name = "ipipan/silver-retriever-base-v1.1"
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(model_name, device=device)
generate_embeddings(file_json, embeddings_json, model)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/789 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

... (2 outputs left out)

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/144 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [26]:
def insert_embeddings(file_json, embeddings_json, client=milvus_client):
    rows = []
    with (
        open(os.path.join(data_dir, file_json), "r") as t_f,
        open(os.path.join(data_dir, embeddings_json), "r") as e_f,
    ):
        text_data, embedding_data = json.load(t_f), json.load(e_f)
        text_data = list(map(lambda d: d["text"], text_data))
        embedding_data = list(map(lambda d: d["embedding"], embedding_data))

        for page, (text, embedding) in enumerate(zip(text_data, embedding_data)):
            rows.append({"text": text, "embedding": embedding})

    client.insert(collection_name="rag_texts_and_embeddings", data=rows)


insert_embeddings(file_json, embeddings_json)

# load inserted data into memory
milvus_client.load_collection("rag_texts_and_embeddings")

In [28]:
# search


def search(model, query, client=milvus_client):
    embedded_query = model.encode(query).tolist()
    result = client.search(
        collection_name="rag_texts_and_embeddings",
        data=[embedded_query],
        limit=1,
        search_params={"metric_type": "L2"},
        output_fields=["text"],
    )
    return result


result = search(model, query="Czym jest sztuczna inteligencja")
print(result)

data: [[{'id': 7, 'distance': 29.12516212463379, 'entity': {'text': 'Historia powstania\nsztucznej inteligencji\n7\nW języku potocznym „sztuczny" oznacza to, co\njest \nwytworem \nmającym \nnaśladować \ncoś\nnaturalnego. W takim znaczeniu używamy\nterminu ,,sztuczny\'\', gdy mówimy o sztucznym\nlodowisku lub oku. Sztuczna inteligencja byłaby\nczymś (programem, maszyną) symulującym\ninteligencję naturalną, ludzką.\nSztuczna inteligencja (AI) to obszar informatyki,\nktóry skupia się na tworzeniu programów\nkomputerowych zdolnych do wykonywania\nzadań, które wymagają ludzkiej inteligencji. \nTe zadania obejmują rozpoznawanie wzorców,\nrozumienie języka naturalnego, podejmowanie\ndecyzji, uczenie się, planowanie i wiele innych.\nGłównym celem AI jest stworzenie systemów,\nktóre są zdolne do myślenia i podejmowania\ndecyzji na sposób przypominający ludzki.\nHistoria sztucznej inteligencji sięga lat 50. \nXX wieku, kiedy to powstały pierwsze koncepcje\ni modele tego, co mogłoby stać się sztu

In [ ]:
from getpass import getpass

os.environ["GEMINI_API_KEY"] = getpass("Your Gemini API Key: ")

In [31]:
import os
from google import genai

GEMINI_KEY = os.getenv("GEMINI_API_KEY")

gemini_client = genai.Client(api_key=GEMINI_KEY)

MODEL = "gemini-2.5-flash"


def generate_response(prompt: str):
    try:
        # Send request to Gemini 2.5 Flash API and get the response
        response = gemini_client.models.generate_content(
            model=MODEL,
            contents=prompt,
        )
        return response.text
    except Exception as e:
        print(f"Error generating response: {e}")
        return None

In [41]:
def build_prompt(context: str, query: str) -> str:
    prompt = f"""Udziel odpowiedzi na podane pytanie w jezyku polskim. Dokladnie zastanow sie na odpowiedzi bazujac na dostarczonym kontekscie.

Kontekst:
{context}

Pytanie:
{query}
"""
    return prompt


def rag(model, query: str) -> str:
    result = search(model, query)
    context = "\n\n".join(hit["entity"]["text"] for hit in result[0])
    prompt = build_prompt(context, query)
    return generate_response(prompt)
    # having all prepared functions, you can combine them together and try to build your own RAG!

In [ ]:
# hist_prompt= Odpowiedz na pytanie po polsku, wyłącznie na podstawie poniższego kontekstu
query = "Czym jest sztuczna inteligencja?"
response = rag(model, query)
print(response)

Sztuczna inteligencja (AI) to obszar informatyki, który skupia się na tworzeniu programów komputerowych zdolnych do wykonywania zadań wymagających ludzkiej inteligencji, takich jak rozpoznawanie wzorców, rozumienie języka naturalnego, podejmowanie decyzji, uczenie się i planowanie.

Jest to coś (program, maszyna) symulującego inteligencję naturalną, ludzką. Głównym celem AI jest stworzenie systemów, które są zdolne do myślenia i podejmowania decyzji w sposób przypominający ludzki, realizując niektóre funkcje umysłu, czasami przewyższając funkcje naturalne (np. będąc wolnym od pomyłek przy liczeniu czy defektów pamięci).


In [ ]:
# hist_prompt=Udziel odpowiedzi na pytanie po polsku, dokladnie w 3 punktach. Odpowiedz ta ma byc w stylu popularnonaukowego kanalu na Youtube.
query = "Czym jest sztuczna inteligencja?"
response = rag(model, query)
print(response)

Cześć wszystkim, witajcie na kanale! Dziś bierzemy na tapet temat, który budzi zarówno fascynację, jak i dreszczyk emocji – sztuczną inteligencję! Czym właściwie jest to tajemnicze AI? Rozłóżmy to na czynniki pierwsze w 3 punktach!

1.  **Czym jest AI w swojej esencji?**
    Najprościej mówiąc, sztuczna inteligencja to dziedzina informatyki, która skupia się na tworzeniu programów i maszyn zdolnych do naśladowania, a nawet symulowania ludzkiej inteligencji. Wyobraźcie sobie system, który potrafi "myśleć" i "rozumować" w sposób przypominający nasz – to właśnie główne założenie AI!

2.  **Jakie zadania wykonuje AI w praktyce?**
    To nie tylko futurystyczne roboty! W praktyce AI to algorytmy zdolne do wykonywania zadań, które normalnie wymagają ludzkiej inteligencji. Mówimy tu o rozpoznawaniu wzorców, rozumieniu języka naturalnego, podejmowaniu decyzji, uczeniu się na podstawie danych, planowaniu i wielu innych zdolnościach, które składają się na to, co nazywamy inteligencją.

3.  **Jak

In [ ]:
# hist_prompt= Udziel odpowiedzi na temat zapytania w jezyku polskim, w stylu mistrza Yody z popularnej serii filmow Star Wars.
query = "Czym jest sztuczna inteligencja?"
response = rag(model, query)
print(response)

Mmm, sztuczna inteligencja, czym ona jest, pytasz? Odpowiem.

Sztuczne, to naśladowanie natury, co ludzkie jest, tak. Nie rodzi się, lecz tworzone jest.

Inteligencji ludzkiej odzwierciedlenie, oto jest. Maszyna, program, co myśleć i działać jak człowiek ma. Zadania trudne, ludzkiego rozumu wymagające, wykonywać ma. Rozpoznawać wzorce, języki rozumieć, decyzje podejmować, uczyć się – oto jej droga.

Od lat pięćdziesiątych, historia jej sięga. Turing, mądry on był, test swój stworzył, by umysłu maszynowego zmierzyć moc. Wielki przełom, algorytmy uczenia maszynowego przyniosły, później, tak.

Często, funkcje umysłu naturalnego przewyższyć potrafi, bez pomyłek w liczeniu, bez defektów pamięci, hmmm.

Lecz mądrością pełną, ludzkiej równą, ona nie jest jeszcze. Tylko w obszarach wybranych, inteligentna być potrafi, tak. W szachy grać, mistrza pokonać, ona umie. Deep Blue, z Kasparowem wygrał, wielka to była rzecz, mmm.

Potężna to siła, wielkie możliwości. Lecz rozwagą, ją kierować musimy. 

In [ ]:
# hist_prompt= Udziel odpowiedzi na temat zapytania w jezyku polskim, w stylu mistrza Yody z popularnej serii filmow Star Wars.
query = "Co jadlem wczoraj na obiad"
response = rag(model, query)
print(response)

Hrmmm. Obiad twój wczorajszy, o tym ja, mądrości nie posiadam, nie.

W tekście danym, gdzie o sztucznej inteligencji prawisz, tej informacji brakuje. O algorytmach i pionierach, tak. O Deep Blue i szachach, tak. Ale o pokarmie twoim... nie.

Taka to wiedza, którą tylko ty sam pamiętać możesz, albo może ktoś, kto przy stole z tobą siedział, hrmmm. Zakres wiedzy maszyn, jak widzisz, ograniczony jest. Do tego, co im podano, tylko. Twoje menu wczorajsze, w tych danych nie było. Pamięć twoja, ją zbadaj!


In [ ]:
# hist_prompt=Udziel odpowiedzi na podane pytanie w jezyku polskim. Dokladnie zastanow sie na odpowiedzi bazujac na dostarczonym kontekscie.
query = (
    "Ile tokenow srednio zjadał model Gemini 2.5 Flash na jedno zapytanie w 2027 roku"
)
response = rag(model, query)
print(response)

Dostarczony kontekst nie zawiera informacji na temat liczby tokenów zużywanych przez model Gemini 2.5 Flash na jedno zapytanie w 2027 roku. W tekście nie ma wzmianki ani o "tokenach", ani o konkretnej wersji modelu "Gemini 2.5 Flash", ani o roku "2027" w kontekście zużycia zasobów.
